In [ ]:
import numpy as np
import time
import cv2
import os
import zlib
from sdlarch_rl import make
from IPython.display import Audio
from stable_baselines3 import PPO
# from sbx import PPO
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback, FrameSkip, TimeLimit
from sdlarch_rl.utils.discretizer import MainDiscretizer
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from pathlib import Path
from stable_baselines3.common.monitor import Monitor
import sys
# from sbx.ppo.policies import CnnPolicy

import logging
import multiprocessing as mp
mp.set_start_method("spawn", force=True)
logging.basicConfig(level=logging.DEBUG)

# try fix pygame crash
os.environ['PYGAME_BLEND_ALPHA_SDL2'] = '0'
os.environ['PYGAME_BLEND_ALPHA_SDL2_RENDER'] = '0'

if 'pygame' in sys.modules:
    del sys.modules['pygame']
    # remove another pygame modules
    pygame_modules = [m for m in sys.modules if m and 'pygame' in m]
    for mod in pygame_modules:
        del sys.modules[mod]

NUM_ENV = 2
SAVE_DIR="./model-nsm"
TENSORBOARD="./tensorboard-nsm"
TOTAL_TIMESTEP_NUMB = 500_000_000
CHECK_FREQ_NUMB = 5_000
SAVE_FREQ = CHECK_FREQ_NUMB
MAX_STEPS= 8_000

ENT_COEF = 0.001
n_steps=4096
batch_size=64 * NUM_ENV

SAVE_DIR = Path(SAVE_DIR)
combos = [
    [],
    # run
    ["LEFT", "X"],
    ["RIGHT", "X"],
    
    # jump and run
    ["Y"],
    ["LEFT", "X", "Y"],
    ["RIGHT", "X", "Y"],
    
    #shake and run
    ["R2"],
    ["LEFT", "X", "R2"],
    ["RIGHT", "X", "R2"],
]


def make_env(env_id):
    def _init():
        env = make(
            "NewSuperMarioBros-Wii", 
            env_id=env_id,
            # render_mode="human"
        )
        env.set_buttons(["B", "Y", "SELECT", "START", "LEFT", "RIGHT", "DOWN", "UP","A", "X", "L1", "R1", "L2", "R2", "L3", "R3"])

        env = Monitor(env)

        env = MainDiscretizer(
            env,
            combos,
        )

        env = WarpFrame(env, width=96, height=96)
        # env = WarpFrame(env)
        # env = MaxAndSkipEnv(env, skip=4)
        env = FrameSkip(env, skip=4)
        env = TimeLimit(env, max_steps=MAX_STEPS)

        return env
    return _init

    
# env = make_vec_env(make_env(), n_envs=NUM_ENV)
envs = [make_env(i) for i in range(NUM_ENV)]
env = SubprocVecEnv(envs)
env = VecFrameStack(env, 4, channels_order='last')

latest_model_path = get_latest_model(SAVE_DIR)

if latest_model_path:
    print(f"Loading existent model: {latest_model_path}")
    model = PPO.load(
    # model = RecurrentPPO.load(
        str(latest_model_path), 
        env=env, 
        verbose=0, 
        tensorboard_log=TENSORBOARD, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
    )
    
else:
    print("None finded, starting from zero.")
    model = PPO("CnnPolicy", 
    # model = RecurrentPPO('CnnLstmPolicy',
        env, 
        verbose=0, 
        # policy_kwargs=policy_kwargs, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
        tensorboard_log=TENSORBOARD, 
    )

# eval_callback = EvalCallback(
#     eval_env, 
#     best_model_save_path="./logs/best_model",
#     log_path="./logs/results", 
#     eval_freq=5_000,
#     n_eval_episodes=6,
#     deterministic=True
# )

checkpoint_callback=TrainAndLoggingCallback(check_freq=CHECK_FREQ_NUMB, save_path=SAVE_DIR, save_freq=SAVE_FREQ, model=model)
#callback = CallbackList([checkpoint_callback, eval_callback])
callback = CallbackList([checkpoint_callback])

model.learn(total_timesteps=TOTAL_TIMESTEP_NUMB, reset_num_timesteps=False, callback=callback)
model.save("final_nsm")

env.close()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


Loading existent model: model-nsm\best_model_1260000
Done Rewards Step Cnt: 746
Done Rewards Step Cnt: 730
Done Rewards Step Cnt: 532
Done Rewards Step Cnt: 381
Done Rewards Step Cnt: 615
Done Rewards Step Cnt: 263
Done Rewards Step Cnt: 258
Done Rewards Step Cnt: 708
Done Rewards Step Cnt: 219
Model saved in: model-nsm\best_model_1265000
Done Rewards Step Cnt: 722
Done Rewards Step Cnt: 195
Done Rewards Step Cnt: 698
Done Rewards Step Cnt: 230
Done Rewards Step Cnt: 580
Done Rewards Step Cnt: 144
Done Rewards Step Cnt: 724
Done Rewards Step Cnt: 420
Done Rewards Step Cnt: 712
Done Rewards Step Cnt: 188
Done Rewards Step Cnt: 521
Done Rewards Step Cnt: 100
Model saved in: model-nsm\best_model_1270000
Done Rewards Step Cnt: 536
Done Rewards Step Cnt: 230
Done Rewards Step Cnt: 144
Done Rewards Step Cnt: 481
Done Rewards Step Cnt: 584
Done Rewards Step Cnt: 237
Done Rewards Step Cnt: 79
Done Rewards Step Cnt: 244
Done Rewards Step Cnt: 684
Done Rewards Step Cnt: 656
Done Rewards Step Cnt